In [1]:
import numpy as np
import pandas as pd
import re
import string
import pickle

In [2]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [3]:
import string

def remove_punctuationstest(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [4]:
with open('../static/model/model.pickle', 'rb') as f:
    model = pickle.load(f)

In [5]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
tokens = vocab[0].tolist()

In [6]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [7]:
def peprocessing(text):
    data = pd.DataFrame([text], columns=['tweet'])
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(word.lower() for word in x.split()))
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','', word, flags=re.MULTILINE) for word in x.split()))
    data['tweet'] = data['tweet'].apply(remove_punctuationstest)
    data['tweet'] = data['tweet'].str.replace(r'\d+', '', regex=True)
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(ps.stem(word) for word in x.split() if word not in sw))    
    return data['tweet']

In [11]:
def vectorizer(ds, vocabulary):
    vectorized_lst = []

    for sentence in ds:
        sentence_lst = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_lst[i] = 1

        vectorized_lst.append(sentence_lst)
        
    vectorized_lst_new = np.asarray(vectorized_lst, dtype=np.float32)
    
    return vectorized_lst_new

In [9]:
def get_prediction(vectorized_text):
    prediction = model.predict(vectorized_text)
    if prediction == 1: 
        return 'negative'
    else:
        return 'positive'

In [10]:
txt = 'awesome product. i love it'
preprocessed_txt = peprocessing(txt)
vectorized_txt = vectorizer(preprocessed_txt, tokens)
prediction = get_prediction(vectorized_txt)
prediction

'positive'